In [1]:
import pandas as pd

print('Setup complete')

Setup complete


In [2]:
# Use the raw tables here so missed calls remain available for call counts.
sms_data = pd.read_csv('../data/raw/sms.csv')
calls_data = pd.read_csv('../data/raw/calls.csv')
bt_data = pd.read_csv('../data/raw/bt_symmetric.csv')
fb_data = pd.read_csv('../data/raw/fb_friends.csv').rename(columns={'# user_a': 'user_a'})

# Synthetic features

## Single-person features implemented below

- Average call time
- Number of texts sent
- Number of texts received
- Number of calls sent
- Number of calls received
- Total interactions received (texts received + calls received)

- Number of unique contacts (text)
- Number of unique contacts (call)
- Signal strength (interaction distance)
- Friendship Concentrations


In [3]:
# Include every participant found in the label or interaction tables.
participant_ids = sorted(
    set(calls_data['caller'])
    | set(calls_data['callee'])
    | set(sms_data['sender'])
    | set(sms_data['recipient'])
)
participant_features = pd.DataFrame({'user': participant_ids})

# Map raw aggregate counts without imputing missing values.
count_features = {
    'texts_sent': sms_data.groupby('sender').size(),
    'texts_received': sms_data.groupby('recipient').size(),
    'calls_sent': calls_data.groupby('caller').size(),
    'calls_received': calls_data.groupby('callee').size(),
}

for feature_name, counts in count_features.items():
    participant_features[feature_name] = participant_features['user'].map(counts)

# Give each participant the duration of every completed call they took part in.
completed_calls = calls_data.loc[calls_data['duration'].ge(0)]
call_durations_by_participant = pd.concat(
    [
        completed_calls[['caller', 'duration']].rename(columns={'caller': 'user'}),
        completed_calls[['callee', 'duration']].rename(columns={'callee': 'user'}),
    ],
    ignore_index=True,
)
average_call_time = call_durations_by_participant.groupby('user')['duration'].mean()
participant_features['average_call_time'] = participant_features['user'].map(
    average_call_time
)

participant_features['total_interactions_received'] = (
    participant_features['texts_received'] + participant_features['calls_received']
)



display(participant_features.head())
display(participant_features.describe())

,user,texts_sent,texts_received,calls_sent,calls_received,average_call_time,total_interactions_received
0,0,62.0,74.0,3.0,6.0,28.444444,80.0
1,1,2.0,1.0,NaN,NaN,NaN,NaN
2,3,147.0,147.0,11.0,10.0,33.750000,157.0
3,4,82.0,69.0,24.0,20.0,30.736842,89.0
4,5,21.0,18.0,1.0,5.0,42.666667,23.0


,user,texts_sent,texts_received,calls_sent,calls_received,average_call_time,total_interactions_received
count,608.000000,555.000000,555.000000,449.000000,480.000000,525.000000,445.000000
mean,378.976974,43.843243,43.843243,8.017817,7.500000,59.933494,58.838202
std,227.522527,126.450568,124.806691,10.487222,9.942003,93.111181,140.172130
min,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,2.000000
25%,181.500000,5.000000,5.000000,2.000000,2.000000,18.368421,11.000000
50%,371.500000,14.000000,15.000000,4.000000,4.000000,33.000000,24.000000
75%,564.750000,36.500000,37.500000,9.000000,9.000000,60.500000,57.000000
max,846.000000,1585.000000,1586.000000,94.000000,101.000000,776.000000,1590.000000


In [4]:
unique_calls = calls_data.groupby("callee").agg(
    calls_unique_values=("caller", "unique"), calls_unique_count=("caller", "nunique")
).reset_index()
display(unique_calls)


participant_features = pd.merge(
    participant_features, unique_calls, left_on="user", right_on="callee", how="left"
)

unique_texts = (
    sms_data.groupby("sender")
    .agg(texts_unique_values=("recipient", "unique"), texts_unique_count=("recipient", "nunique"))
    .reset_index()
)
display(unique_texts)


participant_features = pd.merge(
    participant_features, unique_texts, left_on="user", right_on="sender", how="left"
)
participant_features


,callee,calls_unique_values,calls_unique_count
0,0,"[512, 208]",2
1,3,"[49, 48, 357, 485]",4
2,4,"[221, 266, 424, 144, 344]",5
3,5,[802],1
4,6,"[27, 406, 616]",3
...,...,...,...
475,812,[179],1
476,813,[671],1
477,830,"[140, 545]",2
478,843,[170],1


,sender,texts_unique_values,texts_unique_count
0,0,"[512, 208]",2
1,1,[345],1
2,3,"[357, 49, 217]",3
3,4,"[266, 221, 344, 424, 381, 507, 176, 144]",8
4,5,"[802, 269]",2
...,...,...,...
550,813,[671],1
551,830,"[140, 257, 118]",3
552,843,[170],1
553,845,"[128, 256]",2


,user,texts_sent,texts_received,calls_sent,calls_received,average_call_time,total_interactions_received,callee,calls_unique_values,calls_unique_count,sender,texts_unique_values,texts_unique_count
0,0,62.0,74.0,3.0,6.0,28.444444,80.0,0.0,"[512, 208]",2.0,0.0,"[512, 208]",2.0
1,1,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,[345],1.0
2,3,147.0,147.0,11.0,10.0,33.750000,157.0,3.0,"[49, 48, 357, 485]",4.0,3.0,"[357, 49, 217]",3.0
3,4,82.0,69.0,24.0,20.0,30.736842,89.0,4.0,"[221, 266, 424, 144, 344]",5.0,4.0,"[266, 221, 344, 424, 381, 507, 176, 144]",8.0
4,5,21.0,18.0,1.0,5.0,42.666667,23.0,5.0,[802],1.0,5.0,"[802, 269]",2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,813,13.0,13.0,NaN,1.0,0.000000,14.0,813.0,[671],1.0,813.0,[671],1.0
604,830,22.0,27.0,4.0,7.0,16.909091,34.0,830.0,"[140, 545]",2.0,830.0,"[140, 257, 118]",3.0
605,843,8.0,10.0,NaN,1.0,0.000000,11.0,843.0,[170],1.0,843.0,[170],1.0
606,845,9.0,7.0,4.0,3.0,43.200000,10.0,845.0,"[256, 703]",2.0,845.0,"[128, 256]",2.0


In [5]:
participant_features["calls_unique_values"] = participant_features[
    "calls_unique_values"
].fillna({i: [] for i in participant_features.index})
participant_features = participant_features.fillna(0)
participant_features

,user,texts_sent,texts_received,calls_sent,calls_received,average_call_time,total_interactions_received,callee,calls_unique_values,calls_unique_count,sender,texts_unique_values,texts_unique_count
0,0,62.0,74.0,3.0,6.0,28.444444,80.0,0.0,"[512, 208]",2.0,0.0,"[512, 208]",2.0
1,1,2.0,1.0,0.0,0.0,0.000000,0.0,0.0,[],0.0,1.0,[345],1.0
2,3,147.0,147.0,11.0,10.0,33.750000,157.0,3.0,"[49, 48, 357, 485]",4.0,3.0,"[357, 49, 217]",3.0
3,4,82.0,69.0,24.0,20.0,30.736842,89.0,4.0,"[221, 266, 424, 144, 344]",5.0,4.0,"[266, 221, 344, 424, 381, 507, 176, 144]",8.0
4,5,21.0,18.0,1.0,5.0,42.666667,23.0,5.0,[802],1.0,5.0,"[802, 269]",2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,813,13.0,13.0,0.0,1.0,0.000000,14.0,813.0,[671],1.0,813.0,[671],1.0
604,830,22.0,27.0,4.0,7.0,16.909091,34.0,830.0,"[140, 545]",2.0,830.0,"[140, 257, 118]",3.0
605,843,8.0,10.0,0.0,1.0,0.000000,11.0,843.0,[170],1.0,843.0,[170],1.0
606,845,9.0,7.0,4.0,3.0,43.200000,10.0,845.0,"[256, 703]",2.0,845.0,"[128, 256]",2.0


In [6]:
participant_output_file = '../data/interim/participant_features.csv'
participant_features.to_csv(participant_output_file, index=False, encoding='utf-8')
print(f'Saved {len(participant_features):,} participant rows to {participant_output_file}')

Saved 608 participant rows to ../data/interim/participant_features.csv


## Pairwise features

Each row represents an **unordered pair**, identified by `user_a < user_b`. The pair universe is the union of valid participant pairs observed in proximity, SMS, calls, or Facebook friendships.

- `average_proximity_rssi`: mean RSSI across valid participant-to-participant Bluetooth measurements. Larger (less negative) values indicate closer proximity.
- `texts_shared`: messages in either direction.
- `calls_shared`: call attempts in either direction, including missed calls.
- `longest_consecutive_days`: longest raw streak of consecutive active days. Normalization is intentionally deferred.
- `average_call_contact_time` and `max_call_contact_time`: completed calls only; missed-call sentinels are excluded.
- `call_reciprocity` and `text_reciprocity`: `min(A→B, B→A) / max(A→B, B→A)`, ranging from 0 (one-way) to 1 (balanced).
- `fraction_rssi_above_threshold` and `fraction_rssi_below_threshold`: fractions above and at/below `RSSI_THRESHOLD = -75` dBm.
- `weekend_interaction_fraction` and `weekday_interaction_fraction`: shares of distinct active pair-days. The observation period begins on Sunday, so elapsed days 0 and 6 modulo 7 are weekends.
- `mutual_friends`: size of the intersection of the participants' Facebook-neighbor sets.

No imputation, scaling, or consecutive-day normalization is applied here. The requested reciprocity and percentage features remain ratios by definition. Missing cross-layer measurements are preserved for later cleaning and modeling decisions. Both feature tables are exported to `data/interim/`.

In [7]:
RSSI_THRESHOLD = -75
SECONDS_PER_DAY = 86_400

def make_undirected_pairs(df, left_col, right_col, timestamp_col=None):
    """Return valid unordered pairs while retaining source direction and time."""
    columns = [left_col, right_col]
    if timestamp_col is not None:
        columns.append(timestamp_col)
    pairs = df[columns].copy()
    left_values = pairs[left_col].copy()
    right_values = pairs[right_col].copy()
    pairs['user_a'] = pd.concat([left_values, right_values], axis=1).min(axis=1)
    pairs['user_b'] = pd.concat([left_values, right_values], axis=1).max(axis=1)
    pairs['from_a'] = left_values.eq(pairs['user_a'])
    return pairs.loc[pairs['user_a'].ge(0) & pairs['user_a'].lt(pairs['user_b'])]


bt_valid = bt_data.loc[bt_data['user_b'].ge(0)].rename(
    columns={'# timestamp': 'timestamp'}
)
bt_pairs = make_undirected_pairs(bt_valid, 'user_a', 'user_b', 'timestamp')
bt_pairs['rssi'] = bt_valid.loc[bt_pairs.index, 'rssi']

sms_pairs = make_undirected_pairs(sms_data, 'sender', 'recipient', 'timestamp')
call_pairs = make_undirected_pairs(calls_data, 'caller', 'callee', 'timestamp')
call_pairs['duration'] = calls_data.loc[call_pairs.index, 'duration']
fb_pairs = make_undirected_pairs(fb_data, 'user_a', 'user_b')

pair_keys = ['user_a', 'user_b']
all_pairs = (
    pd.concat(
        [bt_pairs[pair_keys], sms_pairs[pair_keys], call_pairs[pair_keys], fb_pairs[pair_keys]],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(pair_keys)
    .reset_index(drop=True)
)

print(f'Pair universe: {len(all_pairs):,} unordered pairs')

Pair universe: 82,851 unordered pairs


In [8]:
bt_pair_features = (
    bt_pairs.groupby(pair_keys)
    .agg(
        average_proximity_rssi=('rssi', 'mean'),
        proximity_measurements=('rssi', 'size'),
        fraction_rssi_above_threshold=('rssi', lambda values: values.gt(RSSI_THRESHOLD).mean()),
        fraction_rssi_below_threshold=('rssi', lambda values: values.le(RSSI_THRESHOLD).mean()),
    )
    .reset_index()
)

sms_pair_features = (
    sms_pairs.groupby(pair_keys).size().rename('texts_shared').reset_index()
)

call_pair_features = (
    call_pairs.groupby(pair_keys).size().rename('calls_shared').reset_index()
)
completed_call_pairs = call_pairs.loc[call_pairs['duration'].ge(0)]
call_duration_features = (
    completed_call_pairs.groupby(pair_keys)
    .agg(
        average_call_contact_time=('duration', 'mean'),
        max_call_contact_time=('duration', 'max'),
    )
    .reset_index()
)


def calculate_reciprocity(directed_pairs, feature_name):
    """Calculate min(direction counts) / max(direction counts) for each pair."""
    direction_counts = (
        directed_pairs.groupby(pair_keys + ['from_a'])
        .size()
        .unstack(fill_value=0)
    )
    from_a = direction_counts.get(True, pd.Series(0, index=direction_counts.index))
    from_b = direction_counts.get(False, pd.Series(0, index=direction_counts.index))
    denominator = pd.concat([from_a, from_b], axis=1).max(axis=1)
    reciprocity = from_a.combine(from_b, min).div(denominator)
    return reciprocity.rename(feature_name).reset_index()


text_reciprocity = calculate_reciprocity(sms_pairs, 'text_reciprocity')
call_reciprocity = calculate_reciprocity(call_pairs, 'call_reciprocity')

In [9]:
# Use distinct active pair-days so five-minute Bluetooth scans do not dominate time shares.
interaction_events = pd.concat(
    [
        bt_pairs[pair_keys + ['timestamp']],
        sms_pairs[pair_keys + ['timestamp']],
        call_pairs[pair_keys + ['timestamp']],
    ],
    ignore_index=True,
)
interaction_events['elapsed_day'] = (
    interaction_events['timestamp'] // SECONDS_PER_DAY
).astype('int64')
active_pair_days = interaction_events[pair_keys + ['elapsed_day']].drop_duplicates()

def longest_consecutive_streak(days):
    ordered_days = pd.Series(days.unique()).sort_values(ignore_index=True)
    streak_groups = ordered_days.diff().ne(1).cumsum()
    return int(ordered_days.groupby(streak_groups).size().max())


pair_time_features = (
    active_pair_days.groupby(pair_keys)
    .agg(
        active_days=('elapsed_day', 'nunique'),
        longest_consecutive_days=('elapsed_day', longest_consecutive_streak),
        weekend_interaction_fraction=(
            'elapsed_day', lambda days: days.mod(7).isin([0, 6]).mean()
        ),
    )
    .reset_index()
)
pair_time_features['weekday_interaction_fraction'] = (
    1.0 - pair_time_features['weekend_interaction_fraction']
)

In [10]:
# Build an undirected Facebook adjacency map, excluding invalid self-edges.
friend_neighbors = {}
for row in fb_pairs.itertuples(index=False):
    friend_neighbors.setdefault(row.user_a, set()).add(row.user_b)
    friend_neighbors.setdefault(row.user_b, set()).add(row.user_a)

mutual_friend_features = all_pairs[pair_keys].copy()
mutual_friend_features['mutual_friends'] = [
    len(friend_neighbors.get(user_a, set()) & friend_neighbors.get(user_b, set()))
    for user_a, user_b in mutual_friend_features[pair_keys].itertuples(index=False, name=None)
]

In [11]:
pairwise_features = all_pairs.copy()
feature_tables = [
    bt_pair_features,
    sms_pair_features,
    call_pair_features,
    call_duration_features,
    text_reciprocity,
    call_reciprocity,
    pair_time_features,
    mutual_friend_features,
]
for feature_table in feature_tables:
    pairwise_features = pairwise_features.merge(feature_table, on=pair_keys, how='left')

# Structural validation only; missing values and raw scales are preserved.
assert pairwise_features[pair_keys].duplicated().sum() == 0
assert pairwise_features['user_a'].lt(pairwise_features['user_b']).all()
bounded_columns = [
    'fraction_rssi_above_threshold', 'fraction_rssi_below_threshold',
    'text_reciprocity', 'call_reciprocity',
    'weekend_interaction_fraction', 'weekday_interaction_fraction',
]
bounded_values = pairwise_features[bounded_columns].stack().dropna().round(12)
assert bounded_values.between(0, 1).all()
rssi_rows = pairwise_features['proximity_measurements'].gt(0)
assert (
    pairwise_features.loc[rssi_rows, 'fraction_rssi_above_threshold']
    + pairwise_features.loc[rssi_rows, 'fraction_rssi_below_threshold']
).round(12).eq(1.0).all()

pairwise_output_file = '../data/interim/pairwise_features.csv'
pairwise_features.to_csv(pairwise_output_file, index=False, encoding='utf-8')
print(f'Saved {len(pairwise_features):,} pair rows to {pairwise_output_file}')
display(pairwise_features.head())
display(pairwise_features.describe())

Saved 82,851 pair rows to ../data/interim/pairwise_features.csv


,user_a,user_b,average_proximity_rssi,proximity_measurements,fraction_rssi_above_threshold,fraction_rssi_below_threshold,texts_shared,calls_shared,average_call_contact_time,max_call_contact_time,text_reciprocity,call_reciprocity,active_days,longest_consecutive_days,weekend_interaction_fraction,weekday_interaction_fraction,mutual_friends
0,0,3,-92.000000,2.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.000000,1.000000,2
1,0,5,-92.750000,4.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.000000,1.000000,1
2,0,10,-96.666667,3.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.000000,1.000000,0
3,0,12,-83.785714,28.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,0.333333,0.666667,1
4,0,13,-94.833333,6.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,0.000000,1.000000,3


,user_a,user_b,average_proximity_rssi,proximity_measurements,fraction_rssi_above_threshold,fraction_rssi_below_threshold,texts_shared,calls_shared,average_call_contact_time,max_call_contact_time,text_reciprocity,call_reciprocity,active_days,longest_consecutive_days,weekend_interaction_fraction,weekday_interaction_fraction,mutual_friends
count,82851.000000,82851.000000,79530.000000,79530.000000,79530.000000,79530.000000,697.000000,621.000000,605.000000,605.000000,697.000000,621.000000,79692.000000,79692.000000,79692.000000,79692.000000,82851.00000
mean,225.810612,460.777323,-89.168558,30.507720,0.055741,0.944259,34.911047,5.797101,52.723227,140.041322,0.658877,0.296143,2.395661,1.283165,0.037514,0.962486,1.02863
std,159.861481,171.835506,5.872091,123.278621,0.150638,0.150638,145.786908,10.104649,84.774737,327.806320,0.321806,0.363104,2.583840,1.030901,0.154556,0.154556,2.54650
min,0.000000,3.000000,-100.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.00000
25%,92.000000,339.000000,-93.250000,1.000000,0.000000,0.979626,3.000000,1.000000,13.333333,23.000000,0.500000,0.000000,1.000000,1.000000,0.000000,1.000000,0.00000
50%,194.000000,481.000000,-90.000000,5.000000,0.000000,1.000000,9.000000,3.000000,29.761905,55.000000,0.750000,0.000000,1.000000,1.000000,0.000000,1.000000,0.00000
75%,336.500000,596.000000,-86.000000,19.000000,0.020374,1.000000,23.000000,6.000000,58.000000,119.000000,0.918919,0.500000,3.000000,1.000000,0.000000,1.000000,1.00000
max,824.000000,850.000000,-36.000000,5766.000000,1.000000,1.000000,2860.000000,130.000000,776.000000,5138.000000,1.000000,1.000000,28.000000,28.000000,1.000000,1.000000,44.00000


## Daily call-volume features

For each pair, attach each member's overall daily call activity — not limited to
calls between the two of them, but their total call behavior with anyone. Counts
are normalized by `TOTAL_OBSERVATION_DAYS`, the length of the study window shared
by the whole cohort (the same fixed window already used for the weekend/weekday
elapsed-day features above).

- `calls_received_per_day_a` / `_b`: all incoming calls (answered or missed).
- `calls_sent_per_day_a` / `_b`: all outgoing calls.
- `missed_calls_per_day_a` / `_b`: incoming calls with `duration == -1`, i.e. calls
  the participant missed. ("Missed call" is an inherently incoming concept, so this
  mirrors the "received ... from anyone" framing rather than outgoing no-answers.)

Unlike the interaction-specific pairwise features above, a missing value here means
the participant truly placed/received zero calls in the study, so these six columns
are filled with 0 rather than left as `NaN`.

In [12]:
# The full cohort is observed over the same fixed window, so a single constant
# denominator is used rather than each participant's own active-day span.
TOTAL_OBSERVATION_DAYS = int(
    pd.concat([calls_data['timestamp'], sms_data['timestamp'], bt_valid['timestamp']]).max()
    // SECONDS_PER_DAY
) + 1

missed_calls = calls_data.loc[calls_data['duration'].eq(-1)]

daily_call_rate_features = pd.DataFrame({'user': participant_ids})
daily_call_rate_features['calls_received_per_day'] = (
    daily_call_rate_features['user'].map(calls_data.groupby('callee').size()).fillna(0)
    / TOTAL_OBSERVATION_DAYS
)
daily_call_rate_features['calls_sent_per_day'] = (
    daily_call_rate_features['user'].map(calls_data.groupby('caller').size()).fillna(0)
    / TOTAL_OBSERVATION_DAYS
)
daily_call_rate_features['missed_calls_per_day'] = (
    daily_call_rate_features['user'].map(missed_calls.groupby('callee').size()).fillna(0)
    / TOTAL_OBSERVATION_DAYS
)

print(f'Observation window: {TOTAL_OBSERVATION_DAYS} days')
display(daily_call_rate_features.head())
display(daily_call_rate_features.describe())

Observation window: 28 days


,user,calls_received_per_day,calls_sent_per_day,missed_calls_per_day
0,0,0.214286,0.107143,0.000000
1,1,0.000000,0.000000,0.000000
2,3,0.357143,0.392857,0.035714
3,4,0.714286,0.857143,0.000000
4,5,0.178571,0.035714,0.107143


,user,calls_received_per_day,calls_sent_per_day,missed_calls_per_day
count,608.000000,608.000000,608.000000,608.000000
mean,378.976974,0.211466,0.211466,0.021499
std,227.522527,0.333818,0.345541,0.052300
min,0.000000,0.000000,0.000000,0.000000
25%,181.500000,0.035714,0.000000,0.000000
50%,371.500000,0.107143,0.071429,0.000000
75%,564.750000,0.250000,0.250000,0.035714
max,846.000000,3.607143,3.357143,0.428571


In [13]:
daily_rate_columns = ['calls_received_per_day', 'calls_sent_per_day', 'missed_calls_per_day']

for id_column, suffix in [('user_a', '_a'), ('user_b', '_b')]:
    renamed = daily_call_rate_features.rename(columns={'user': id_column}).rename(
        columns={col: f'{col}{suffix}' for col in daily_rate_columns}
    )
    pairwise_features = pairwise_features.merge(renamed, on=id_column, how='left')

suffixed_rate_columns = [f'{col}{suffix}' for col in daily_rate_columns for suffix in ('_a', '_b')]
pairwise_features[suffixed_rate_columns] = pairwise_features[suffixed_rate_columns].fillna(0)

pairwise_features.to_csv(pairwise_output_file, index=False, encoding='utf-8')
print(f'Re-saved {len(pairwise_features):,} pair rows with daily call-rate features to {pairwise_output_file}')
display(pairwise_features[pair_keys + suffixed_rate_columns].head())

Re-saved 82,851 pair rows with daily call-rate features to ../data/interim/pairwise_features.csv


,user_a,user_b,calls_received_per_day_a,calls_received_per_day_b,calls_sent_per_day_a,calls_sent_per_day_b,missed_calls_per_day_a,missed_calls_per_day_b
0,0,3,0.214286,0.357143,0.107143,0.392857,0.0,0.035714
1,0,5,0.214286,0.178571,0.107143,0.035714,0.0,0.107143
2,0,10,0.214286,0.000000,0.107143,0.000000,0.0,0.000000
3,0,12,0.214286,0.321429,0.107143,0.464286,0.0,0.000000
4,0,13,0.214286,0.750000,0.107143,0.750000,0.0,0.285714
